## Secret Memory
---

- Purpose: Creates a memory-backed file descriptor with enhanced security properties,
designed to mitigate speculative execution attacks (e.g., Spectre) by preventing the
kernel from accessing the memory contents after the initial mapping.

- Security Note: The memory region is unmapped from kernel pages after mapping,
so only the user process can access it, reducing exposure to side-channel attacks.

In [2]:
:dep libc

In [3]:
use libc::{syscall, SYS_memfd_secret, mmap, PROT_READ, PROT_WRITE, MAP_SHARED, MAP_FAILED};
use std::{fs::File, io::Write, os::{fd::{AsRawFd, FromRawFd, OwnedFd}, unix::io::FromRawFd}, ptr, process::Command};

In [4]:
// Linux syscall #447 is secret memory.
SYS_memfd_secret

447

In [5]:
// Checking if Secret Memeory is enabled is NOT this simple.
String::from_utf8(Command::new("rg").args(["SECRETMEM", "/boot/config-6.17.0-8-generic"]).output()?.stdout)?

"CONFIG_SECRETMEM=y\n"

In [6]:
// Create the secret file descriptor.
let secret_fd = unsafe { syscall(SYS_memfd_secret, 0) };
if secret_fd < 0 {
    panic!("Secret memfd failed. Note: Requires Linux 5.14+ and 'memfd_secret' enabled in boot cmdline.");
}
let secret_file_descriptor = unsafe { OwnedFd::from_raw_fd(secret_fd as i32) };
secret_file_descriptor

OwnedFd { fd: 3 }

In [7]:
// Set the size (memfd_secret starts at 0 bytes).
let page_size = unsafe { libc::sysconf(libc::_SC_PAGESIZE) };
let size = (1024 * page_size) as usize;
unsafe { libc::ftruncate(secret_file_descriptor.as_raw_fd(), size as i64) };
println!("{} / {}", page_size, size);

4096 / 4194304


In [8]:
// Map it into virtual address space.
let secret_address = unsafe {
    mmap(
        ptr::null_mut(),
        size,
        PROT_READ | PROT_WRITE,
        MAP_SHARED,
        secret_file_descriptor.as_raw_fd(),
        0,
    )
};
if secret_address == MAP_FAILED {
    panic!("mmap failed");
}
secret_address

0x7207a2200000

In [9]:
// Write directly to the memory pointer.
unsafe {
    let msg = b"Hello world from secret memory!";
    ptr::copy_nonoverlapping(msg.as_ptr(), secret_address as *mut u8, msg.len());
    
// Read it back out.
    let slice = std::slice::from_raw_parts(secret_address as *const u8, msg.len());
    println!("Secret Memory Contents: {}", String::from_utf8_lossy(slice));
};

Secret Memory Contents: Hello world from secret memory!


In [10]:
// Ask the kernel to write to secret memory.
let mut secret_file = unsafe { File::from_raw_fd(secret_file_descriptor.as_raw_fd()) };
secret_file.write(b"Can the kernel do this?")

Err(Os { code: 22, kind: InvalidInput, message: "Invalid argument" })

In [11]:
// 6. Cleanup: unmap when done
unsafe { libc::munmap(secret_address, size); };